# Notebook 2 of 2 — my real parts

You have printed and measured all eight test pieces. This builds the enclosure
sized for **your** printer.

1. Paste your **final** code below — it starts `S1U-`.
2. Click **Runtime → Run all**.
3. Wait about fifteen minutes; the download starts on its own.

If your code starts `S1U1-` you are in the wrong notebook: that is the round one
code, and it only knows your printer's scale. Use **notebook 1** to get the
remaining test pieces first.


In [ ]:
#@title Step 1 — paste your final code here, then press the ▶ button { display-mode: "form" }
MY_CODE = ""  #@param {type:"string"}

import base64

# Two notebooks, two codes, and each refuses the other's. Getting this wrong is
# not a small mistake: a round-one code carries scale and nothing else, so
# building the enclosure from it would size every hole, seat and gasket gap to
# a value nobody has measured yet.
WANT = "S1U-"
OTHER = "S1U1-"
OTHER_NAME = "round one"

KEYS = ["xy_scale_correction_fraction","z_scale_correction_fraction","fastener_clearance_diameter_offset_mm","insert_bore_diameter_offset_mm","driver_cutout_diameter_offset_mm","passive_radiator_cutout_diameter_offset_mm","cable_passage_diameter_offset_mm","gasket_sheet_thickness_mm","gasket_compressed_thickness_offset_mm","active_driver_flange_thickness_mm","passive_radiator_flange_thickness_mm"]

text = MY_CODE.strip()
if not text:
    raise SystemExit("Paste your code into the box above, then run this cell again.")
if text.startswith(OTHER):
    raise SystemExit(
        f"That is a {OTHER_NAME} code. This notebook wants one starting {WANT}.\n"
        "Open the other notebook, or go back to the website's Calibrate step.")
if not text.startswith(WANT):
    raise SystemExit(f"That does not look like a code. It should start with {WANT}")

payload = text[len(WANT):]
try:
    numbers = [float(v) for v in base64.b64decode(payload + "=" * (-len(payload) % 4)).decode().split(",")]
except Exception:
    raise SystemExit("That code looks damaged. Copy it again from the website.")
if len(numbers) != len(KEYS):
    raise SystemExit("That code is incomplete. Copy it again from the website.")

CALIBRATION = dict(zip(KEYS, numbers))
print("Your measurements were read correctly:\n")
for key, value in CALIBRATION.items():
    print(f"  {key:45s} {value}")


## Everything below runs on its own

You do not need to change anything here.


In [ ]:
#@title Install the CAD engine (about 3 minutes)
# numpy is held below 2.1 to match the numba that Colab pre-installs. Nothing
# here imports numba, so the mismatch was harmless, but pip printed it in red as
# an ERROR and that is not a thing to show someone halfway through their first
# print. cadquery declares no numpy requirement of its own, and the arrays used
# in this project are elementary, so the older line is equivalent for our
# purposes. If Colab's numba moves, this pin can move with it.
# This list is not a guess: it is every non-stdlib package reachable from the
# imports this notebook makes, and tests/test_notebook.py walks that import
# chain and fails if the two ever disagree. trimesh went missing when the
# editable install was removed, because that install was the only thing that
# had ever mentioned the project's dependencies, and the failure landed in the
# build cell rather than here.
!pip install --quiet cadquery==2.6.1 pyyaml==6.0.2 trimesh==4.6.10 "numpy<2.1" "scipy<1.16" 2>&1 | tail -2
print("CAD engine ready.")


In [ ]:
#@title Fetch the Satellite1 Ultra design
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path("/content/Satellite1-Ultra")

# This used to run `pip install -e .`, which refuses on any Python outside the
# >=3.12,<3.13 pin in pyproject.toml. That pin describes the development and CI
# environment, not the source: every module here parses as 3.11. Colab moves its
# Python from time to time, so on the wrong day the install failed, the failure
# was piped through `tail -1` and hidden, and the next cell died with a bare
# "No module named 'satellite1_ultra'" that says nothing about the real cause.
#
# The package is pure Python, so putting src on the path is equivalent, cannot
# fail for a version reason, and needs no build backend.
shutil.rmtree(REPO, ignore_errors=True)
clone = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/BigPappy098/Satellite1-Ultra.git", str(REPO)],
    capture_output=True, text=True,
)
if clone.returncode != 0:
    raise SystemExit("Could not download the design files:\n" + clone.stderr[-900:])

os.chdir(REPO)                       # the config file is written relative to here
sys.path.insert(0, str(REPO / "src"))

try:
    import satellite1_ultra  # noqa: F401
    from satellite1_ultra.configuration import ROOT
except Exception as error:
    raise SystemExit(
        f"The design files downloaded but would not load on Python "
        f"{sys.version.split()[0]}.\n{type(error).__name__}: {error}"
    )

assert ROOT == REPO, f"the package resolved its root to {ROOT}, not {REPO}"
print(f"Design files ready. Python {sys.version.split()[0]}.")


In [ ]:
#@title Check your numbers are safe
from satellite1_ultra.configuration import validate_physical_calibration

validate_physical_calibration(CALIBRATION)   # refuses anything physically implausible

with open("config/physical_calibration.yaml", "w") as handle:
    handle.write("# Generated from your measurements.\n")
    for key, value in CALIBRATION.items():
        handle.write(f"{key}: {value}\n")
print("Your numbers passed every safety check.")


In [ ]:
#@title Build your parts (about 10 minutes — this is the slow one)
import time
from pathlib import Path
from satellite1_ultra.configuration import load_design_parameters
from satellite1_ultra.exporting import export_parts

start = time.time()
parameters = load_design_parameters()
written = export_parts(Path("exports"), parameters)
print(f"\nBuilt {len(written)} files in {(time.time()-start)/60:.1f} minutes.")


In [ ]:
#@title Package your parts and download them
import shutil, os
from pathlib import Path
from satellite1_ultra.builder_files import ULTRA_PRINT_ORDER, OFFICIAL_TOP_PRINT_ORDER
from satellite1_ultra.official import OFFICIAL_PRINT_PARTS_REQUIRED

out = Path("/content/MY_SATELLITE1_ULTRA_PARTS")
shutil.rmtree(out, ignore_errors=True)

(out / "1_ENCLOSURE_PARTS").mkdir(parents=True, exist_ok=True)
for source, friendly, _quantity in ULTRA_PRINT_ORDER:
    shutil.copy2(Path("exports/3mf") / f"{source}.3mf", out / "1_ENCLOSURE_PARTS" / friendly)
    shutil.copy2(Path("exports/stl") / f"{source}.stl",
                 out / "1_ENCLOSURE_PARTS" / f"{Path(friendly).stem}.stl")

# The six Satellite top parts are the official files, copied unchanged.
official = {part.name: part for part in OFFICIAL_PRINT_PARTS_REQUIRED}
(out / "2_SATELLITE_TOP_PARTS").mkdir(parents=True, exist_ok=True)
for source, friendly, _quantity in OFFICIAL_TOP_PRINT_ORDER:
    shutil.copy2(official[source].stl_path, out / "2_SATELLITE_TOP_PARTS" / friendly)

(out / "READ_ME.txt").write_text(
    "Your parts, sized for your printer from the measurements you entered.\n\n"
    "1_ENCLOSURE_PARTS      - the enclosure. Print every file.\n"
    "2_SATELLITE_TOP_PARTS  - the original Satellite1 top. Print all six.\n\n"
    "Then go back to the website and follow the Build it step.\n")

archive = shutil.make_archive("/content/MY_SATELLITE1_ULTRA_PARTS", "zip", out.parent, out.name)
print(f"Ready: {os.path.getsize(archive)/1e6:.1f} MB")
try:
    from google.colab import files
    files.download(archive)
    print("\nYour download should start now.")
except Exception:
    print("\nOpen the folder icon on the left and download MY_SATELLITE1_ULTRA_PARTS.zip")


## Done

Your parts are in **MY_SATELLITE1_ULTRA_PARTS.zip**.

If the download did not start, click the folder icon on the left and download it
from there.

Now go back to the website and follow **Build it**.

*These files are generated from your measurements. Nothing has been physically
built and tested yet, so keep checking as you go.*
